In [5]:
%sql 

select * from dev.spark_db.flight_time limit(3)

,FL_DATE,OP_CARRIER,OP_CARRIER_FL_NUM,ORIGIN,ORIGIN_CITY_NAME,DEST,DEST_CITY_NAME,CRS_DEP_TIME,DEP_TIME,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,CANCELLED,DISTANCE
0,2000-01-01,DL,1451,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 11:15:00,0 days 11:13:00,1343,0 days 00:05:00,0 days 14:00:00,0 days 13:48:00,0,946
1,2000-01-01,DL,1479,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 13:15:00,0 days 13:11:00,1536,0 days 00:07:00,0 days 15:59:00,0 days 15:43:00,0,946
2,2000-01-01,DL,1857,BOS,"Boston, MA",ATL,"Atlanta, GA",0 days 14:15:00,0 days 14:14:00,1642,0 days 00:09:00,0 days 17:21:00,0 days 16:51:00,0,946


Requirements are
Read data from flight_time and transform as below
1. Rename fl_date to dep_date
2. Compute arr_date
3. Following fields to represent full timestamp
    1. crs_dep_time
    2. dep_time
    3. crs_arr_time
    4. arr_time

In [12]:
flight_time_df = spark.read.table("dev.spark_db.flight_time")

flight_time_1_df = (
    flight_time_df.selectExpr(
        # 1. Rename fl_date to dep_date
        "fl_date as dep_date",

        # 2. Compute arr_date: same day as the departure,
        #    or the next day if the plane lands after midnight (arrival time earlier than departure time)
        "case when arr_time < dep_time then date_add(fl_date, 1) else fl_date end as arr_date",

        # 3. Full timestamps: date + interval hour to minute = timestamp
        "fl_date + crs_dep_time as crs_dep_time",
        "fl_date + dep_time as dep_time",
        "arr_date + crs_arr_time as crs_arr_time",
        "arr_date + arr_time as arr_time",

        "op_carrier",
        "op_carrier_fl_num"     # needed by the where() below
    )
)

# display(df), not df.display(): df.display() only exists inside Databricks notebooks
display(flight_time_1_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'"))

,dep_date,arr_date,crs_dep_time,dep_time,crs_arr_time,arr_time,op_carrier,op_carrier_fl_num
0,2000-01-01,2000-01-01,2000-01-01 11:15:00,2000-01-01 11:13:00,2000-01-01 14:00:00,2000-01-01 13:48:00,DL,1451
1,2000-01-01,2000-01-01,2000-01-01 18:50:00,2000-01-01 19:42:00,2000-01-01 21:55:00,2000-01-01 22:33:00,AA,1451
2,2000-01-01,2000-01-01,2000-01-01 09:40:00,2000-01-01 09:40:00,2000-01-01 10:35:00,2000-01-01 10:37:00,WN,1451


#selectexpr() is not the best way to do this transformation as it will lose the rest of the columns in the data frame as if we wanted to see the rest of the table we will have to add it manually like op_carrier & op_carrier_fl_num so its better to use withcolumns() so that we dont lose other fields.

### selectExpr() vs withColumns()

- `selectExpr("expr as name", ...)`: returns **only** the listed columns. Takes SQL strings and can rename with `as`.
  A column made in the call can be used by later ones (e.g. `arr_date`).
- `withColumns({"name": col, ...})`: keeps **all** columns, adds new ones or replaces existing ones (same name).
  Each expression only sees the **original** columns.

In [ ]:
#doing it using withColumns()

from pyspark.sql.functions import expr

flight_time_2_df = (
  flight_time_df.withColumnRenamed("fl_date", "dep_date")
      # arr_date: next day if the plane lands after midnight (arrival time earlier than departure time)
      .withColumn("arr_date", expr("case when arr_time < dep_time then date_add(dep_date, 1) else dep_date end"))
      .withColumns({
        "crs_dep_time": expr("dep_date + crs_dep_time"),
        "dep_time": expr("dep_date + dep_time"),
        "crs_arr_time": expr("arr_date + crs_arr_time"),
        "arr_time": expr("arr_date + arr_time"),
      })
)

display(flight_time_2_df.where("op_carrier_fl_num = 1451 and dep_date = '2000-01-01'"))

### Alternative: Column objects instead of SQL strings

Build expressions from Python objects (`col("x") + col("y")`, `when()`, `date_add()`) instead of SQL strings (`expr("...")`, `selectExpr(...)`).
Both compile to the **same plan**, so performance is identical. The difference is how easy the code is to write and maintain.

**Why use Column objects**
1. **Errors show up sooner**: a typo inside `expr("dep_dat + dep_time")` is only caught when Spark parses the string. With Column objects the editor catches syntax errors and autocompletes functions.
2. **Works well with Python code**: expressions can be built in loops or functions, e.g. `{c: col("dep_date") + col(c) for c in time_cols}`, without fragile f-strings.
3. **Reusable**: keep an expression in a variable (`is_overnight = col("arr_time") < col("dep_time")`) and use it in `withColumn`, `where` and `select`.
4. **Awkward column names**: `col("my col")` works without SQL backtick escaping.
5. **Documented and typed**: everything in `pyspark.sql.functions` has docstrings and type hints.

**When SQL strings are the better choice**
- The logic reads more naturally as SQL (`case when ... end` is shorter than `when().otherwise()`).
- Copying logic from an existing SQL query.
- Readers are more comfortable with SQL than with the PySpark API.

Both styles can be mixed in the same chain.

> ⚠️ The code below computes `arr_date` as `to_date(dep_date + dep_time + wheels_on + taxi_in)`, which is different from the `case when` version above.
> `wheels_on` is a clock time (when the wheels touch down), not a duration, so adding it on top of `dep_time` may count time twice. Compare the results with the `case when` version.

In [ ]:
# Same transformation as above, but written with Column objects instead of SQL strings.
# col("name") refers to a column; +, ==, & etc. build expressions that Spark evaluates per row.
from pyspark.sql.functions import to_date, col

flight_time_2_df = (
  flight_time_df
      # 1. Rename fl_date to dep_date (all other columns are kept as they are)
      .withColumnRenamed("fl_date", "dep_date")

      # 2. Compute arr_date: start from the departure date, add the time-of-day/interval columns
      #    to get a timestamp, then to_date() drops the time part and keeps only the date.
      #    ⚠️ wheels_on is a clock time, not a duration, so this may count time twice (see note above).
      .withColumn("arr_date", to_date(col("dep_date") + col("dep_time") + col("wheels_on") + col("taxi_in")))

      # 3. Turn the time columns into full timestamps: date + interval (hour to minute) = timestamp.
      #    withColumns() replaces columns that already exist with the same name.
      #    Departure times use dep_date; arrival times use arr_date (created in the previous step,
      #    which is why it is a separate withColumn() call: expressions inside one withColumns()
      #    only see the columns that existed before that call).
      .withColumns({
        "crs_dep_time": col("dep_date") + col("crs_dep_time"),   # scheduled departure
        "dep_time": col("dep_date") + col("dep_time"),           # actual departure
        "crs_arr_time": col("arr_date") + col("crs_arr_time"),   # scheduled arrival
        "arr_time": col("arr_date") + col("arr_time"),           # actual arrival
      })
)

# Check one flight. With Column objects:
#   - use == for equality (not =)
#   - combine conditions with & (and), | (or), ~ (not)
#   - wrap each condition in parentheses, because & is evaluated before ==
flight_time_2_df.where((col("op_carrier_fl_num") == 1451) & (col("dep_date") == '2000-01-01')).display()